# 🏢 Microsoft Supply Chain Network Optimization — Graph Algorithms

**Session 2 | Industrial Use Case 1 | DSA & ML for Business**

---

### Business Context
- Microsoft's hardware supply chain: **Surface, Xbox, Azure servers, HoloLens**
- Components move between **30+ factories & 60+ data centers** globally
- COVID-19 exposed fragility: **64% of companies** faced supply disruption in 2021
- Goal: minimize cost while maintaining delivery SLAs and managing risk

### What You'll Learn
1. Build a **multi-layer supply chain graph** (Supplier → Factory → Warehouse → Data Center)
2. Compute **shortest paths** (Dijkstra) optimizing for cost and time
3. Calculate **Minimum Spanning Tree** for minimum-cost network
4. Use **betweenness centrality** to identify critical vulnerability nodes
5. **Simulate node failure** and quantify disruption impact

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load dataset
df = pd.read_csv("../datasets/microsoft_supply_chain.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Classify node types
def get_node_type(name):
    if 'Supplier' in name: return 'Supplier'
    elif 'Factory' in name: return 'Factory'
    elif 'Warehouse' in name: return 'Warehouse'
    elif 'DC' in name: return 'Data Center'
    return 'Unknown'

all_nodes = set(df['source']) | set(df['destination'])
node_types = {n: get_node_type(n) for n in all_nodes}

print(f"\nNode types:")
for t in ['Supplier', 'Factory', 'Warehouse', 'Data Center']:
    count = sum(1 for v in node_types.values() if v == t)
    print(f"  {t}: {count}")
print(f"\nTotal edges: {len(df)}")
df.head(10)

## Step 2: Build the Supply Chain Graph

In [ ]:
# Build directed weighted graph
G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_edge(row['source'], row['destination'],
               cost=row['cost'],
               delivery_time=row['delivery_time'],
               risk_score=row['risk_score'],
               capacity=row['capacity'])

# Set node attributes
nx.set_node_attributes(G, node_types, 'type')

print(f"Supply Chain Graph:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")

# Visualize the supply chain
fig, ax = plt.subplots(figsize=(16, 10))

layer_order = {'Supplier': 0, 'Factory': 1, 'Warehouse': 2, 'Data Center': 3}
pos = {}
for layer_name, layer_x in layer_order.items():
    nodes_in_layer = [n for n, t in node_types.items() if t == layer_name]
    for i, node in enumerate(sorted(nodes_in_layer)):
        pos[node] = (layer_x * 3, -(i - len(nodes_in_layer)/2) * 1.2)

type_colors = {'Supplier': '#EF4444', 'Factory': '#F59E0B', 'Warehouse': '#10B981', 'Data Center': '#2563EB'}
node_colors = [type_colors[node_types[n]] for n in G.nodes()]

edge_widths = [G[u][v]['capacity'] / 5000 for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, alpha=0.3, width=edge_widths, edge_color='gray',
                       arrows=True, arrowsize=15, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800,
                       edgecolors='white', linewidths=2, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=6, font_weight='bold', ax=ax)

patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=10)
ax.set_title('Microsoft Supply Chain Network\n(Edge width = Capacity)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3: Shortest Path Analysis (Dijkstra) — Optimizing for Cost & Time

In [ ]:
# Find shortest paths from each supplier to each data center
suppliers = [n for n, t in node_types.items() if t == 'Supplier']
datacenters = [n for n, t in node_types.items() if t == 'Data Center']

print("=== Shortest Paths: Supplier → Data Center (by Cost) ===\n")
all_paths = []
for supplier in sorted(suppliers):
    for dc in sorted(datacenters):
        try:
            path = nx.dijkstra_path(G, supplier, dc, weight='cost')
            cost = nx.dijkstra_path_length(G, supplier, dc, weight='cost')
            total_time = sum(G[path[i]][path[i+1]]['delivery_time'] for i in range(len(path)-1))
            total_risk = max(G[path[i]][path[i+1]]['risk_score'] for i in range(len(path)-1))
            all_paths.append({
                'source': supplier, 'dest': dc,
                'path': ' → '.join([n.split('_')[-1] for n in path]),
                'total_cost': round(cost, 2),
                'total_time': round(total_time, 1),
                'max_risk': round(total_risk, 2),
                'hops': len(path) - 1
            })
        except nx.NetworkXNoPath:
            pass

paths_df = pd.DataFrame(all_paths)
print(f"Total reachable routes: {len(paths_df)}")
print(f"\n--- Cheapest 10 Routes ---")
print(paths_df.nsmallest(10, 'total_cost')[
    ['source', 'dest', 'total_cost', 'total_time', 'max_risk', 'hops']].to_string(index=False))

## Step 4: Minimum Spanning Tree & Centrality Analysis

In [ ]:
# Convert to undirected for MST and centrality
G_undirected = G.to_undirected()

mst = nx.minimum_spanning_tree(G_undirected, weight='cost')
mst_cost = sum(d['cost'] for _, _, d in mst.edges(data=True))
total_cost = sum(d['cost'] for _, _, d in G_undirected.edges(data=True))
print(f"Minimum Spanning Tree:")
print(f"  MST edges: {mst.number_of_edges()} (out of {G.number_of_edges()} total)")
print(f"  MST total cost: ${mst_cost:,.2f}")
print(f"  Full network cost: ${total_cost:,.2f}")
print(f"  Cost savings: ${total_cost - mst_cost:,.2f} ({(1 - mst_cost/total_cost)*100:.1f}%)")

betweenness = nx.betweenness_centrality(G, weight='cost')
print(f"\n=== Betweenness Centrality: Most Critical Nodes ===")
sorted_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)
for node, bc in sorted_betweenness[:10]:
    print(f"  {node:<35s} | Betweenness: {bc:.4f} | Type: {node_types[node]}")

fig, ax = plt.subplots(figsize=(12, 6))
nodes_sorted = sorted(betweenness.keys(), key=lambda x: betweenness[x], reverse=True)[:12]
colors = [type_colors[node_types[n]] for n in nodes_sorted]
ax.barh(range(len(nodes_sorted)), [betweenness[n] for n in nodes_sorted], color=colors)
ax.set_yticks(range(len(nodes_sorted)))
ax.set_yticklabels([n.replace('_', '\n') for n in nodes_sorted], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Betweenness Centrality')
ax.set_title('Most Critical Supply Chain Nodes', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Node Failure Simulation — Disruption Impact Analysis

What happens when the most critical node goes offline?

In [ ]:
# Simulate failure of top critical nodes
print("=== NODE FAILURE SIMULATION ===\n")

baseline_paths = 0
for s in suppliers:
    for dc in datacenters:
        if nx.has_path(G, s, dc):
            baseline_paths += 1

print(f"Baseline reachable routes (Supplier → DC): {baseline_paths}")

critical_nodes = [n for n, _ in sorted_betweenness[:5]]
disruption_results = []

for node in critical_nodes:
    G_fail = G.copy()
    G_fail.remove_node(node)
    remaining_paths = 0
    cost_increase_paths = []
    for s in suppliers:
        for dc in datacenters:
            if s == node or dc == node:
                continue
            try:
                new_cost = nx.dijkstra_path_length(G_fail, s, dc, weight='cost')
                try:
                    old_cost = nx.dijkstra_path_length(G, s, dc, weight='cost')
                    cost_increase_paths.append(new_cost - old_cost)
                except: pass
                remaining_paths += 1
            except nx.NetworkXNoPath:
                pass

    path_loss = baseline_paths - remaining_paths
    avg_cost_increase = np.mean(cost_increase_paths) if cost_increase_paths else 0
    disruption_results.append({
        'node': node, 'type': node_types[node],
        'betweenness': betweenness[node],
        'paths_lost': path_loss,
        'pct_loss': path_loss / baseline_paths * 100,
        'avg_cost_increase': avg_cost_increase
    })
    print(f"\n  ❌ Remove: {node}")
    print(f"     Type: {node_types[node]} | Betweenness: {betweenness[node]:.4f}")
    print(f"     Paths lost: {path_loss}/{baseline_paths} ({path_loss/baseline_paths*100:.1f}%)")
    print(f"     Avg cost increase on remaining routes: ${avg_cost_increase:.2f}")

disrupt_df = pd.DataFrame(disruption_results)

## Step 6: Strategic Analysis & Recommendations

In [ ]:
# Executive Disruption Risk Report
print("=" * 70)
print("EXECUTIVE BRIEF: Supply Chain Disruption Risk Assessment")
print("=" * 70)

worst_case = disrupt_df.nlargest(1, 'pct_loss').iloc[0]
print(f"\n1. CRITICAL VULNERABILITY:")
print(f"   Node: {worst_case['node']}")
print(f"   Type: {worst_case['type']}")
print(f"   Impact if offline: {worst_case['pct_loss']:.1f}% of routes lost")
print(f"   Avg cost increase on surviving routes: ${worst_case['avg_cost_increase']:.2f}/unit")

daily_shipments = 10000
avg_margin_per_unit = 150
delayed_pct = worst_case['pct_loss'] / 100
daily_exposure = daily_shipments * delayed_pct * avg_margin_per_unit

print(f"\n2. FINANCIAL EXPOSURE:")
print(f"   Daily shipments affected: {daily_shipments * delayed_pct:,.0f}")
print(f"   Daily revenue at risk: ${daily_exposure:,.0f}")
print(f"   Monthly exposure: ${daily_exposure * 30:,.0f}")
print(f"   Annual exposure: ${daily_exposure * 365:,.0f}")

print(f"\n3. REDUNDANCY RECOMMENDATION:")
print(f"   → Establish backup route bypassing {worst_case['node']}")
print(f"   → Estimated CapEx for redundancy: ${daily_exposure * 30 * 0.2:,.0f}")
print(f"   → Priority: Add secondary warehouse in same region")

print(f"\n4. MST INSIGHT:")
print(f"   Minimum viable network cost: ${mst_cost:,.2f}")
print(f"   Current redundancy investment: ${total_cost - mst_cost:,.2f}")
print(f"   Redundancy ratio: {(total_cost/mst_cost - 1)*100:.1f}% above minimum")